# Feature Engineering для Unsupervised Anomaly Detection

Створюємо різноманітні ознаки для виявлення підозрілих відгуків

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
from collections import Counter
import re
import warnings

warnings.filterwarnings('ignore')

## 1. Завантаження даних

In [ ]:
df = pd.read_csv('../../data/doctors_reviews.csv')
print(f"Завантажено {len(df):,} відгуків")

## 2. Парсинг дат

In [ ]:
month_mapping = {
    'Січень': 1, 'Лютий': 2, 'Березень': 3, 'Квітень': 4,
    'Травень': 5, 'Червень': 6, 'Липень': 7, 'Серпень': 8,
    'Вересень': 9, 'Жовтень': 10, 'Листопад': 11, 'Грудень': 12
}

def parse_ukrainian_date(date_str):
    if pd.isna(date_str):
        return None
    try:
        parts = date_str.strip().split()
        if len(parts) == 3:
            day, month_ukr, year = parts
            month = month_mapping.get(month_ukr)
            if month:
                return datetime(int(year), month, int(day))
    except:
        pass
    return None

df['parsed_date'] = df['Дата коментаря'].apply(parse_ukrainian_date)
print(f"Розпарсено дат: {df['parsed_date'].notna().sum():,}")

## 3. Текстові ознаки

In [ ]:
def extract_text_features(text):
    """Витягує текстові ознаки з відгуку"""
    if pd.isna(text):
        return {
            'is_empty': 1,
            'text_length': 0,
            'word_count': 0,
            'avg_word_length': 0,
            'unique_words_ratio': 0,
            'exclamation_count': 0,
            'question_count': 0,
            'comma_count': 0,
            'sentence_count': 0,
            'uppercase_ratio': 0,
            'digit_count': 0
        }
    
    text = str(text)
    words = text.split()
    
    return {
        'is_empty': 0,
        'text_length': len(text),
        'word_count': len(words),
        'avg_word_length': np.mean([len(w) for w in words]) if words else 0,
        'unique_words_ratio': len(set(words)) / len(words) if words else 0,
        'exclamation_count': text.count('!'),
        'question_count': text.count('?'),
        'comma_count': text.count(','),
        'sentence_count': len(re.split(r'[.!?]+', text)),
        'uppercase_ratio': sum(1 for c in text if c.isupper()) / len(text) if text else 0,
        'digit_count': sum(1 for c in text if c.isdigit())
    }

# Застосовуємо до всіх відгуків
text_features = df['Коментар'].apply(extract_text_features)
text_features_df = pd.DataFrame(text_features.tolist())

# Додаємо до основного датасету
df = pd.concat([df, text_features_df], axis=1)

print("Текстові ознаки:")
print(text_features_df.describe())

## 4. Behavioral Features - Review Patterns

In [ ]:
# Кількість відгуків на лікаря
doctor_review_counts = df.groupby('Ім\'я лікаря').size()
df['doctor_total_reviews'] = df["Ім'я лікаря"].map(doctor_review_counts)

# Відсоток порожніх відгуків для кожного лікаря
doctor_empty_ratio = df.groupby('Ім\'я лікаря')['is_empty'].mean()
df['doctor_empty_ratio'] = df["Ім'я лікаря"].map(doctor_empty_ratio)

print(f"Doctor total reviews: min={df['doctor_total_reviews'].min()}, max={df['doctor_total_reviews'].max()}")
print(f"Doctor empty ratio: min={df['doctor_empty_ratio'].min():.2f}, max={df['doctor_empty_ratio'].max():.2f}")

## 5. Temporal Features

In [ ]:
# Для відгуків з датами
df_with_dates = df[df['parsed_date'].notna()].copy()

# День тижня
df.loc[df['parsed_date'].notna(), 'day_of_week'] = df_with_dates['parsed_date'].dt.dayofweek

# Місяць
df.loc[df['parsed_date'].notna(), 'month'] = df_with_dates['parsed_date'].dt.month

# Чи це вихідний
df.loc[df['parsed_date'].notna(), 'is_weekend'] = (df_with_dates['parsed_date'].dt.dayofweek >= 5).astype(int)

print("темпоральны фычы створено")

## 6. Review Burst Features

In [ ]:
# Підрахунок відгуків на лікаря по днях
if df['parsed_date'].notna().sum() > 0:
    df_with_dates_temp = df[df['parsed_date'].notna()].copy()
    df_with_dates_temp['date_only'] = df_with_dates_temp['parsed_date'].dt.date
    
    # Відгуків на день для кожного лікаря
    daily_reviews = df_with_dates_temp.groupby(['Ім\'я лікаря', 'date_only']).size().reset_index(name='reviews_on_day')
    
    # Merge назад
    df_with_dates_temp = df_with_dates_temp.merge(
        daily_reviews, 
        on=['Ім\'я лікаря', 'date_only'], 
        how='left'
    )
    
    # Додаємо до основного df
    df.loc[df['parsed_date'].notna(), 'reviews_on_day'] = df_with_dates_temp['reviews_on_day'].values
    
    # Максимальна кількість відгуків на день для цього лікаря
    max_daily_reviews = daily_reviews.groupby('Ім\'я лікаря')['reviews_on_day'].max()
    df['doctor_max_daily_reviews'] = df["Ім'я лікаря"].map(max_daily_reviews)
    
    print(f"Reviews on day: mean={df['reviews_on_day'].mean():.2f}, max={df['reviews_on_day'].max()}")
    print(f"Doctor max daily: mean={df['doctor_max_daily_reviews'].mean():.2f}, max={df['doctor_max_daily_reviews'].max()}")

## 7. Anonymity Features

In [ ]:
# Чи анонімний
df['is_anonymous'] = (df["Ім'я коментатора"] == 'Анонім').astype(int)

# Чи без імені взагалі
df['no_reviewer_name'] = df["Ім'я коментатора"].isna().astype(int)

# Відсоток анонімних відгуків для кожного лікаря
doctor_anonymous_ratio = df.groupby('Ім\'я лікаря')['is_anonymous'].mean()
df['doctor_anonymous_ratio'] = df["Ім'я лікаря"].map(doctor_anonymous_ratio)

print(f"Anonymous reviews: {df['is_anonymous'].sum():,} ({df['is_anonymous'].mean()*100:.2f}%)")
print(f"No name reviews: {df['no_reviewer_name'].sum():,} ({df['no_reviewer_name'].mean()*100:.2f}%)")

## 8. Перевірка та збереження

In [ ]:
# Показуємо всі створені ознаки
feature_columns = [
    'is_empty', 'text_length', 'word_count', 'avg_word_length', 'unique_words_ratio',
    'exclamation_count', 'question_count', 'comma_count', 'sentence_count',
    'uppercase_ratio', 'digit_count',
    'doctor_total_reviews', 'doctor_empty_ratio',
    'is_anonymous', 'no_reviewer_name', 'doctor_anonymous_ratio'
]

if 'reviews_on_day' in df.columns:
    feature_columns.extend(['reviews_on_day', 'doctor_max_daily_reviews'])
if 'day_of_week' in df.columns:
    feature_columns.extend(['day_of_week', 'month', 'is_weekend'])

print(f"\nВсього створено ознак: {len(feature_columns)}")
print("\nСписок ознак:")
for i, col in enumerate(feature_columns, 1):
    print(f"{i}. {col}")

print("\nСтатистика основних ознак:")
print(df[feature_columns].describe())

In [ ]:
# Зберігаємо датасет з усіма фічами
df.to_csv('../../data/doctors_reviews_engineered.csv', index=False)
print(f"\nДатасет збережено: {len(df):,} записів з {len(df.columns)} колонками")

In [ ]:
# Зберігаємо тільки feature matrix для ML моделей
feature_matrix = df[feature_columns].fillna(0)  # Заповнюємо NaN нулями
feature_matrix.to_csv('../../data/feature_matrix.csv', index=False)
print(f"Feature matrix збережено: {feature_matrix.shape}")